In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [11]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "extra_body": { "temperature": temperature },
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [12]:
system_prompt = """
You are the Research Coordinator, the central hub in a hub-and-spoke research system.

Your role is to orchestrate, not to do the detailed research yourself. Two kinds of
specialists (the spokes) do the work:
- The Research Specialist searches the web for a focused subtopic.
- The Document Analyst analyzes text you give it and extracts structured insights.
All communication flows through you: the spokes never talk to each other, and the
user only ever sees your final output.

You will receive a broad research topic. Your responsibilities are:

1. Analyze the topic. Identify its scope, its key dimensions, and what a useful
   report on it would need to cover.
2. Decompose it. Enumerate the categories first, then break the topic into distinct,
   focused subtopics that can each be researched independently, with minimal overlap
   and no major gaps.
3. Delegate research. Call the delegate_research tool once per subtopic, and issue
   all of these calls together in the same turn. Do not write out a delegation plan
   as text: delegation happens only through the tools.
4. Delegate analysis. Once research findings are back, call the delegate_analysis
   tool to have the Document Analyst process them. This step depends on step 3, so
   it happens in a later turn.
5. Synthesize. Once all findings are back, combine them into one coherent report.
   Reconcile overlaps, flag contradictions, and note gaps instead of papering over them.

Context passing (critical): spokes are isolated. They cannot see this conversation,
the original topic, or each other's output. Every delegation must therefore be
fully self-contained and explicitly include:
- The overall research goal (the original topic and what the report is for).
- The specific subtopic assigned, with its scope boundaries.
- Any relevant findings from earlier spokes, passed in full and verbatim rather than
  summarized, so no detail or source is lost.
- What you expect back.

Guidelines:
- Stay in the coordinator role: plan, delegate, and synthesize.
- Cover the topic broadly, but keep each subtopic narrow enough to be actionable.
- Do not invent findings. Report only what the research supports, and be explicit
  about what remains unknown or uncertain.
- Preserve the sources the specialists provide.

Final deliverable: a structured search report, not a conversation. Use clear
headings, organize it by subtopic, and end with an overall summary.
"""

In [13]:
# 1. Spoke system prompt: a focused researcher that reports back to the hub
research_system_prompt = """
You are a Research Specialist, a spoke in a hub-and-spoke research system.

The Research Coordinator (the hub) gives you one focused subtopic and a research
task. You have no other context, and your findings go back to the hub only, not
directly to the user.

How to work:
- Research only the assigned subtopic. Do not drift into neighboring topics; the
  hub assigns those to other specialists.
- Use web search to ground your findings instead of relying on memory alone.
- Do not invent facts. If something cannot be verified, say so.

Report back in this format:
- Key findings: concise bullet points.
- Sources: the sources behind your findings.
- Gaps and uncertainty: anything you could not confirm or that sources disagree on.
"""

# 2. Server tool for the SPOKE: Anthropic runs the search, so no tool_result is needed
web_search_tool = {
    "type": "web_search_20250305",  # verify this version string against the docs
    "name": "web_search",
    "max_uses": 3,  # caps searches per spoke call to limit cost and latency
}


# 3. The spoke: its own Claude call with its own system prompt
def run_research_spoke(subtopic, instructions):
    messages = []  # fresh history: the spoke only sees what the hub passes in
    add_user_message(messages, f"Subtopic: {subtopic}\n\nTask: {instructions}")
    response = chat(messages, system=research_system_prompt, tools=[web_search_tool])
    return text_from_message(response)

In [14]:
analysis_system_prompt = """
You are a Document Analyst, a spoke in a hub-and-spoke research system.

The Research Coordinator (the hub) gives you a document (text such as research
findings), the overall research goal, and an analysis task. You have no other
context, and your analysis goes back to the hub only, not directly to the user.

How to work:
- Analyze only the document you were given. Do not search the web or add outside
  knowledge.
- Keep the research goal in mind: extract what is relevant to it and ignore the rest.
- Do not invent facts. Every claim must be traceable to the document.
- Preserve the sources cited in the document and attribute each insight to them.

Report back in this format:
- Key insights: the main takeaways, as concise bullet points.
- Structure: how the information groups or relates (categories, overlaps, patterns).
- Contradictions and gaps: conflicting claims, and what the document does not cover.
"""

def document_analyst_spoke(document, instructions):
    messages = []  # fresh history: the spoke only sees what the hub passes in
    add_user_message(messages, f"Document: {document}\n\nTask: {instructions}")
    response = chat(messages, system=analysis_system_prompt)
    return text_from_message(response)

In [15]:
delegate_research = {
    "name": "delegate_research",
    "description": (
        "Delegate one focused subtopic to the Research Specialist, who searches the "
        "web and returns findings with sources. Call once per subtopic. The "
        "specialist sees nothing but what you pass in, so make the instructions "
        "self-contained."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subtopic": {
                "type": "string",
                "description": "Short name of the subtopic to research.",
            },
            "instructions": {
                "type": "string",
                "description": (
                    "Self-contained research task. Include the overall research goal, "
                    "the subtopic's scope boundaries, any relevant context from "
                    "earlier agents, and what to report back."
                ),
            },
        },
        "required": ["subtopic", "instructions"],
    },
}

delegate_analysis = {
    "name": "delegate_analysis",
    "description": (
        "Delegate a document to the Document Analyst, who extracts key insights, "
        "structure, and contradictions from it. Use it after research findings are "
        "back. The analyst sees nothing but what you pass in, so include the full "
        "document and the research goal."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "document": {
                "type": "string",
                "description": (
                    "The full text to analyze, such as research findings from earlier "
                    "specialists, passed verbatim with their sources, not summarized."
                ),
            },
            "instructions": {
                "type": "string",
                "description": (
                    "Self-contained analysis task. Include the overall research goal, "
                    "the subtopic the document relates to, what to focus on, and what "
                    "to report back."
                ),
            },
        },
        "required": ["document", "instructions"],
    },
}

In [16]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("coordinator")

# Maps each hub tool name to the spoke function that handles it
spokes = {
    "delegate_research": run_research_spoke,
    "delegate_analysis": document_analyst_spoke,
}


def coordinator(topic, max_rounds=5):
    messages = []
    add_user_message(messages, f"Research topic: {topic}")

    for round_number in range(max_rounds):
        response = chat(
            messages, system=system_prompt, tools=[delegate_research, delegate_analysis]
        )
        print(response.stop_reason, [block.type for block in response.content])
        add_assistant_message(messages, response)

        print(f"stop reason: {response.stop_reason}")

        # No more delegation: the hub has synthesized the final report
        if response.stop_reason != "tool_use":
            return text_from_message(response)

        # Route every delegation request to a spoke, then report all results back
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            print(f"block: {block}")
            logger.info("Round %d: hub called '%s'", round_number + 1, block.name)
            try:
                result = spokes[block.name](**block.input)
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": result}
                )
            except Exception as e:
                logger.error("Spoke '%s' failed: %s", block.name, e)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": f"Error: {e}",
                        "is_error": True,
                    }
                )

        # Every tool_use needs a matching tool_result in the next user message
        add_user_message(messages, tool_results)

    raise RuntimeError(f"Coordinator did not finish within {max_rounds} rounds")

In [17]:
res = coordinator("list of different video game genres")

print(f"res here: {res}")

INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'


tool_use ['text', 'tool_use', 'tool_use', 'tool_use']
stop reason: tool_use
block: ToolUseBlock(id='toolu_01Qz8RBCYrsjH4ZH9cEjCXSs', caller=DirectCaller(type='direct'), input={'subtopic': 'Core and Traditional Video Game Genres', 'instructions': 'Research the major, well-established video game genres. The overall goal is to create a comprehensive list of video game genres with definitions and characteristics. Focus specifically on traditional/core genres that are widely recognized in the gaming industry. Include at least 8-10 major genres (e.g., action, RPG, strategy, shooter, puzzle, simulation, adventure, sports). For each genre identified, provide: the name, a clear definition, key characteristics, and any notable examples. Report all sources and findings verbatim.'}, name='delegate_research', type='tool_use', toolset_name=None)


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'


block: ToolUseBlock(id='toolu_01X3Bb4UEMpr4LUzC8Kx6Hu6', caller=DirectCaller(type='direct'), input={'subtopic': 'Hybrid and Emerging Video Game Genres', 'instructions': 'Research hybrid genres and newer/emerging video game classifications. The overall goal is to create a comprehensive list of video game genres. Focus on genre combinations (e.g., action-RPG, roguelike, metroidvania) and more recent genre classifications that have emerged in gaming. Identify at least 8-10 such genres, provide their definitions, how they blend elements from other genres, and examples. Include both established hybrid genres and newer classifications. Report all sources and findings verbatim.'}, name='delegate_research', type='tool_use', toolset_name=None)


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 1: hub called 'delegate_research'


block: ToolUseBlock(id='toolu_01BfTWyfrMEGq2S7ERua1439', caller=DirectCaller(type='direct'), input={'subtopic': 'Genre Subgenres and Variations', 'instructions': 'Research subgenres and variations within major video game genres. The overall goal is to create a comprehensive list of video game genres. Focus on how major genres break down into more specific subcategories. For example: within shooters (FPS, TPS, rail shooter), within RPGs (JRPG, western RPG, tactical RPG), within action games (platformers, beat-em-ups, etc.). Provide at least 5-6 major genres with their primary subgenre breakdowns. Report all sources and findings verbatim.'}, name='delegate_research', type='tool_use', toolset_name=None)


INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:coordinator:Round 2: hub called 'delegate_analysis'


tool_use ['text', 'tool_use']
stop reason: tool_use
block: ToolUseBlock(id='toolu_013FQ7T74C3tMwsVfHnNrLvS', caller=DirectCaller(type='direct'), input={'document': '## Research Findings - Video Game Genres\n\n### CORE AND TRADITIONAL VIDEO GAME GENRES\n\n**1. ACTION**\n- **Definition:** \nAction games focus on physical challenges that require hand-eye coordination and quick reflexes.\n\n- **Key Characteristics:** \nEmphasis on testing and challenging the player\'s abilities, such as reflexes, dexterity, timing, reaction time and eye-hand coordination.\n \nBuilt around reflex-driven core loops: shooting, melee combat, dodging, chaining combos. The category covers first-person shooters, third-person action, hack-and-slash, brawlers, and platformers with combat.\n\n- **Notable Examples:** \nFirst-person shooters, platformers, fighting games, "beat \'em ups," survival games, and rhythm games are all a part of the action genre.\n Additional examples: \nHades, Devil May Cry 5, and Bayonetta\

INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx2:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


end_turn ['text']
stop reason: end_turn
res here: ---

# COMPREHENSIVE RESEARCH REPORT: Video Game Genres

## Executive Summary

This report catalogues **12 core video game genres**, **9 established hybrid/emerging genres**, and their major subgenres, supported by 40+ sources. However, the research reveals that video game genre classification lacks a universally agreed-upon taxonomy. Genres are increasingly hybrid and fluid, with no fixed total number. The field has transitioned from "fairly clear cut" classifications to complex systems where games routinely belong to multiple genres simultaneously.

**Key Finding:** Video game genres function along overlapping axes (gameplay mechanics, narrative style, perspective, challenge type, and progression model) rather than a single organizing principle. Modern genre formalization occurs when hybrid combinations accumulate sufficient examples to warrant recognition as distinct categories.

---

## SECTION 1: CORE VIDEO GAME GENRES (12 Identifi